In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(r"C:\Users\ssaty\OneDrive\Desktop\AI_Personal_Finance\data\Indian_Personal_FinanceandSpending_Habits_data.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df.columns.tolist()

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum().sum()

In [ ]:
data = df.drop_duplicates().reset_index(drop=True)

In [ ]:
data.shape

In [ ]:
expense_columns = [
    "Rent",
    "Loan_Repayment",
    "Insurance",
    "Groceries",
    "Transport",
    "Eating_Out",
    "Entertainment",
    "Utilities",
    "Healthcare",
    "Education",
    "Miscellaneous"
]

In [ ]:
data["calculated_expenses"] = data[expense_columns].sum(axis=1)

In [ ]:
data["expense_ratio"] = (
    data["calculated_expenses"] / data["Income"]
)

In [ ]:
data["loan_payment_ratio"] = (
    data["Loan_Repayment"] / data["Income"]
)

In [ ]:
data["savings_ratio"] = (
    data["Desired_Savings"] / data["Income"]
)

In [ ]:
data["financial_surplus_ratio"] = (
    data["Disposable_Income"] / data["Income"]
)

In [ ]:
data[
    [
        "Income",
        "calculated_expenses",
        "expense_ratio",
        "loan_payment_ratio",
        "savings_ratio",
        "financial_surplus_ratio"
    ]
].head()

In [ ]:
data[
    [
        "expense_ratio",
        "loan_payment_ratio",
        "savings_ratio",
        "financial_surplus_ratio"
    ]
].describe()

In [ ]:
(data["financial_surplus_ratio"] < 0).sum()

In [ ]:
def create_financial_health(row):
    
    # Needs Improvement
    if (
        row["financial_surplus_ratio"] < 0
        or
        (
            row["expense_ratio"] > 0.90
            and row["savings_ratio"] < 0.07
        )
    ):
        return "Needs Improvement"
    
    # Healthy
    elif (
        row["expense_ratio"] < 0.70
        and
        row["savings_ratio"] >= 0.10
        and
        row["financial_surplus_ratio"] >= 0.25
    ):
        return "Healthy"
    
    # Moderate
    else:
        return "Moderate"

In [ ]:
data["Financial_Health"] = data.apply(
    create_financial_health,
    axis=1
)

In [ ]:
data["Financial_Health"].value_counts()

In [ ]:
data["Financial_Health"].value_counts(normalize=True) * 100

In [ ]:
features = [
    "Income",
    "Age",
    "Dependents",
    "City_Tier",
    "Rent",
    "Loan_Repayment",
    "Insurance",
    "Groceries",
    "Transport",
    "Eating_Out",
    "Entertainment",
    "Utilities",
    "Healthcare",
    "Education",
    "Miscellaneous",
    "loan_payment_ratio"
]

In [ ]:
X = data[features]

In [ ]:
y = data["Financial_Health"]

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
print(y_train.shape)
print(y_test.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [ ]:
categorical_features = ["City_Tier"]

In [ ]:
preprocessor = ColumnTransformer(
    transformers = [
        ( "city", OneHotEncoder(handle_unknown='ignore'),categorical_features)
    ],remainder="passthrough"
)

In [ ]:
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

In [ ]:
print(X_train_encoded.shape)
print(X_test_encoded.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,random_state=42,class_weight="balanced",
)

In [ ]:
model.fit(X_train_encoded,y_train)

In [ ]:
y_pred = model.predict(X_test_encoded)

In [ ]:
print(y_pred[:10])

In [ ]:
print("Actual:")
print(y_test.iloc[:10].values)

In [ ]:
print("\nPredicted:")
print(y_pred[:10])

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
print("Accuracy:",accuracy_score(y_test,y_pred))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay

In [ ]:
cm = confusion_matrix(y_test,y_pred)

In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=model.classes_ 
)

In [ ]:
disp.plot()
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(X_train_encoded, y_train)

In [ ]:
y_pred_logistic = logistic_model.predict(X_test_encoded)

print("Accuracy:", accuracy_score(y_test, y_pred_logistic))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_logistic))

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train_encoded, y_train)

In [ ]:
y_pred_gb = gb_model.predict(X_test_encoded)

print("Accuracy:", accuracy_score(y_test, y_pred_gb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_gb)) 

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

dt_model.fit(X_train_encoded, y_train)

In [ ]:
y_pred_dt = dt_model.predict(X_test_encoded)

print("Accuracy:", accuracy_score(y_test, y_pred_dt))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

In [ ]:
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

In [ ]:
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_encoded, y_train)

In [ ]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Macro F1:")
print(grid_search.best_score_)

In [ ]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Macro F1:")
print(grid_search.best_score_)

In [ ]:
y_pred_best_rf = best_rf.predict(X_test_encoded)

print("Accuracy:", accuracy_score(y_test, y_pred_best_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_rf))

In [ ]:
import pandas as pd

feature_names = preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": best_rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

In [ ]:
print(feature_importance.head(10))

In [ ]:
print("Features used by the model:")
for i, feature in enumerate(X.columns, start=1):
    print(i, feature)

In [ ]:
features = [
    "Income",
    "Age",
    "Dependents",
    "City_Tier",
    "Rent",
    "Loan_Repayment",
    "Insurance",
    "Groceries",
    "Transport",
    "Eating_Out",
    "Entertainment",
    "Utilities",
    "Healthcare",
    "Education",
    "Miscellaneous"
]

X = data[features]
y = data["Financial_Health"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures:")
print(X.columns.tolist())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["City_Tier"]

preprocessor = ColumnTransformer(
    transformers=[
        ("city", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Encoded X_train shape:", X_train_encoded.shape)
print("Encoded X_test shape :", X_test_encoded.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train_encoded, y_train)

y_pred_rf = rf_model.predict(X_test_encoded)

print("Random Forest training completed.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_rf = accuracy_score(y_test, y_pred_rf)

print("Random Forest Accuracy:", accuracy_rf)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train_encoded, y_train)

y_pred_gb = gb_model.predict(X_test_encoded)

print("Gradient Boosting training completed.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_gb = accuracy_score(y_test, y_pred_gb)

print("Gradient Boosting Accuracy:", accuracy_gb)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_gb))

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(X_train_encoded, y_train)

y_pred_logistic = logistic_model.predict(X_test_encoded)

print("Logistic Regression training completed.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_logistic = accuracy_score(y_test, y_pred_logistic)

print("Logistic Regression Accuracy:", accuracy_logistic)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_logistic))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

dt_model.fit(X_train_encoded, y_train)

y_pred_dt = dt_model.predict(X_test_encoded)

print("Decision Tree training completed.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_dt = accuracy_score(y_test, y_pred_dt)

print("Decision Tree Accuracy:", accuracy_dt)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_encoded, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation Macro F1:")
print(grid_search.best_score_)

In [ ]:
best_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train_encoded, y_train)

y_pred_best_rf = best_rf.predict(X_test_encoded)

print("Tuned Random Forest training completed.")

In [ ]:
best_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train_encoded, y_train)

y_pred_best_rf = best_rf.predict(X_test_encoded)

print("Tuned Random Forest training completed.")

In [ ]:
import pandas as pd

feature_names = preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": best_rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

In [ ]:
import pandas as pd

feature_names = preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": best_rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

In [ ]:
data["Total_Expenses"] = data["calculated_expenses"]

print(data[[
    "Income",
    "Total_Expenses",
    "Loan_Repayment",
    "Desired_Savings",
    "Age",
    "Dependents",
    "City_Tier"
]].head())

In [ ]:
simple_features = [
    "Age",
    "Dependents",
    "City_Tier",
    "Income",
    "Total_Expenses",
    "Loan_Repayment",
    "Desired_Savings"
]

X_simple = data[simple_features]
y_simple = data["Financial_Health"]

print("X shape:", X_simple.shape)
print("y shape:", y_simple.shape)

print("\nFeatures:")
print(X_simple.head())

print("\nTarget:")
print(y_simple.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_simple,
    y_simple,
    test_size=0.20,
    random_state=42,
    stratify=y_simple
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Categorical feature
categorical_features = ["City_Tier"]

# Create preprocessing object
preprocessor_simple = ColumnTransformer(
    transformers=[
        (
            "city",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

# Fit only on training data
X_train_encoded = preprocessor_simple.fit_transform(X_train)

# Transform test data using the same fitted preprocessor
X_test_encoded = preprocessor_simple.transform(X_test)

print("X_train_encoded shape:", X_train_encoded.shape)
print("X_test_encoded shape:", X_test_encoded.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_simple = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_simple.fit(X_train_encoded, y_train)

print("Random Forest training completed.")


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_simple = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_simple.fit(X_train_encoded, y_train)

print("Random Forest training completed.")

In [ ]:
y_pred_rf_simple = rf_simple.predict(X_test_encoded)

print("Prediction completed.")
print("Number of predictions:", len(y_pred_rf_simple))

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_rf_simple = accuracy_score(
    y_test,
    y_pred_rf_simple
)

print("Random Forest Accuracy:", accuracy_rf_simple)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_rf_simple
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(
    y_test,
    y_pred_rf_simple,
    labels=[
        "Healthy",
        "Moderate",
        "Needs Improvement"
    ]
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual Healthy",
        "Actual Moderate",
        "Actual Needs Improvement"
    ],
    columns=[
        "Predicted Healthy",
        "Predicted Moderate",
        "Predicted Needs Improvement"
    ]
)

print(cm_df)

In [ ]:
from sklearn.metrics import f1_score

macro_f1_rf_simple = f1_score(
    y_test,
    y_pred_rf_simple,
    average="macro"
)

print("Macro F1 Score:", macro_f1_rf_simple)

In [ ]:
feature_names = preprocessor_simple.get_feature_names_out()

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_simple.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

print(feature_importance)

In [ ]:
print(feature_importance.head(10))

In [ ]:
print("========== RANDOM FOREST RESULTS ==========")
print(f"Accuracy : {accuracy_rf_simple:.4f}")
print(f"Macro F1 : {macro_f1_rf_simple:.4f}")

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred_rf_simple
))

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(
    y_test,
    y_pred_rf_simple,
    labels=[
        "Healthy",
        "Moderate",
        "Needs Improvement"
    ]
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual Healthy",
        "Actual Moderate",
        "Actual Needs Improvement"
    ],
    columns=[
        "Predicted Healthy",
        "Predicted Moderate",
        "Predicted Needs Improvement"
    ]
)

print(cm_df)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score

gb_simple = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_simple.fit(X_train_encoded, y_train)

y_pred_gb_simple = gb_simple.predict(X_test_encoded)

gb_accuracy = accuracy_score(
    y_test,
    y_pred_gb_simple
)

gb_macro_f1 = f1_score(
    y_test,
    y_pred_gb_simple,
    average="macro"
)

print("Gradient Boosting")
print("-----------------")
print(f"Accuracy : {gb_accuracy:.4f}")
print(f"Macro F1 : {gb_macro_f1:.4f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

logistic_simple = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logistic_simple.fit(X_train_encoded, y_train)

y_pred_logistic_simple = logistic_simple.predict(
    X_test_encoded
)

logistic_accuracy = accuracy_score(
    y_test,
    y_pred_logistic_simple
)

logistic_macro_f1 = f1_score(
    y_test,
    y_pred_logistic_simple,
    average="macro"
)

print("Logistic Regression")
print("-------------------")
print(f"Accuracy : {logistic_accuracy:.4f}")
print(f"Macro F1 : {logistic_macro_f1:.4f}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

dt_simple = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

dt_simple.fit(X_train_encoded, y_train)

y_pred_dt_simple = dt_simple.predict(
    X_test_encoded
)

dt_accuracy = accuracy_score(
    y_test,
    y_pred_dt_simple
)

dt_macro_f1 = f1_score(
    y_test,
    y_pred_dt_simple,
    average="macro"
)

print("Decision Tree")
print("-------------")
print(f"Accuracy : {dt_accuracy:.4f}")
print(f"Macro F1 : {dt_macro_f1:.4f}")

In [ ]:
model_comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Gradient Boosting",
        "Logistic Regression",
        "Decision Tree"
    ],
    "Accuracy": [
        accuracy_rf_simple,
        gb_accuracy,
        logistic_accuracy,
        dt_accuracy
    ],
    "Macro F1": [
        macro_f1_rf_simple,
        gb_macro_f1,
        logistic_macro_f1,
        dt_macro_f1
    ]
})

model_comparison = model_comparison.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

print(model_comparison)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

gb = GradientBoostingClassifier(
    random_state=42
)

param_grid_gb = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [2, 3],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

grid_search_gb = GridSearchCV(
    estimator=gb,
    param_grid=param_grid_gb,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

grid_search_gb.fit(
    X_train_encoded,
    y_train
)

print("Best Parameters:")
print(grid_search_gb.best_params_)

print("\nBest Cross-Validation Macro F1:")
print(grid_search_gb.best_score_)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Create tuned Gradient Boosting model
best_gb = GradientBoostingClassifier(
    learning_rate=0.1,
    max_depth=3,
    min_samples_leaf=2,
    min_samples_split=5,
    n_estimators=200,
    random_state=42
)

# Train on training data
best_gb.fit(
    X_train_encoded,
    y_train
)

# Predict on untouched test data
y_pred_best_gb = best_gb.predict(X_test_encoded)

# Accuracy
best_gb_accuracy = accuracy_score(
    y_test,
    y_pred_best_gb
)

# Macro F1
best_gb_macro_f1 = f1_score(
    y_test,
    y_pred_best_gb,
    average="macro"
)

print("Tuned Gradient Boosting Results")
print("--------------------------------")
print("Accuracy:", best_gb_accuracy)
print("Macro F1:", best_gb_macro_f1)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_best_gb
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred_best_gb
    )
)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Create tuned Gradient Boosting model
best_gb = GradientBoostingClassifier(
    learning_rate=0.1,
    max_depth=3,
    min_samples_leaf=2,
    min_samples_split=5,
    n_estimators=200,
    random_state=42
)

# Train on training data
best_gb.fit(
    X_train_encoded,
    y_train
)

# Predict on untouched test data
y_pred_best_gb = best_gb.predict(X_test_encoded)

# Accuracy
best_gb_accuracy = accuracy_score(
    y_test,
    y_pred_best_gb
)

# Macro F1
best_gb_macro_f1 = f1_score(
    y_test,
    y_pred_best_gb,
    average="macro"
)

print("Tuned Gradient Boosting Results")
print("--------------------------------")
print("Accuracy:", best_gb_accuracy)
print("Macro F1:", best_gb_macro_f1)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_best_gb
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred_best_gb
    )
)

In [ ]:
new_user = pd.DataFrame({
    "Age": [25],
    "Dependents": [2],
    "City_Tier": ["Tier_2"],
    "Income": [40000],
    "Total_Expenses": [28000],
    "Loan_Repayment": [5000],
    "Desired_Savings": [5000]
})

In [ ]:
new_user

In [ ]:
new_user_encoded = preprocessor_simple.transform(new_user)

print("Encoded shape:", new_user_encoded.shape)

In [ ]:
new_user

In [ ]:
new_prediction = best_gb.predict(new_user_encoded)

print("Predicted Financial Health:")
print(new_prediction[0])

In [ ]:
new_probability = best_gb.predict_proba(new_user_encoded)

print("Prediction Probabilities:")

for class_name, probability in zip(
    best_gb.classes_,
    new_probability[0]
):
    print(f"{class_name}: {probability:.2%}")

In [ ]:
# Create 3 different test users

test_users = pd.DataFrame({
    "Age": [30, 35, 28],
    "Dependents": [1, 2, 3],
    "City_Tier": ["Tier_1", "Tier_2", "Tier_3"],
    "Income": [100000, 50000, 30000],
    "Total_Expenses": [50000, 38000, 33000],
    "Loan_Repayment": [5000, 6000, 8000],
    "Desired_Savings": [15000, 4000, 1000]
})

# Apply the same preprocessing
test_users_encoded = preprocessor_simple.transform(test_users)

# Predict financial health
test_predictions = best_gb.predict(test_users_encoded)

# Get probabilities
test_probabilities = best_gb.predict_proba(test_users_encoded)

# Display results
for i in range(len(test_users)):
    
    print(f"\nUser {i + 1}")
    print("-" * 30)
    
    print("Income:", test_users.iloc[i]["Income"])
    print("Total Expenses:", test_users.iloc[i]["Total_Expenses"])
    print("Loan Repayment:", test_users.iloc[i]["Loan_Repayment"])
    print("Desired Savings:", test_users.iloc[i]["Desired_Savings"])
    
    print("Predicted Financial Health:", test_predictions[i])
    
    print("Probabilities:")
    
    for class_name, probability in zip(
        best_gb.classes_,
        test_probabilities[i]
    ):
        print(f"  {class_name}: {probability:.2%}")

In [ ]:
# Create 3 different test users

test_users = pd.DataFrame({
    "Age": [30, 35, 28],
    "Dependents": [1, 2, 3],
    "City_Tier": ["Tier_1", "Tier_2", "Tier_3"],
    "Income": [100000, 50000, 30000],
    "Total_Expenses": [50000, 38000, 33000],
    "Loan_Repayment": [5000, 6000, 8000],
    "Desired_Savings": [15000, 4000, 1000]
})

# Apply the same preprocessing
test_users_encoded = preprocessor_simple.transform(test_users)

# Predict financial health
test_predictions = best_gb.predict(test_users_encoded)

# Get probabilities
test_probabilities = best_gb.predict_proba(test_users_encoded)

# Display results
for i in range(len(test_users)):
    
    print(f"\nUser {i + 1}")
    print("-" * 30)
    
    print("Income:", test_users.iloc[i]["Income"])
    print("Total Expenses:", test_users.iloc[i]["Total_Expenses"])
    print("Loan Repayment:", test_users.iloc[i]["Loan_Repayment"])
    print("Desired Savings:", test_users.iloc[i]["Desired_Savings"])
    
    print("Predicted Financial Health:", test_predictions[i])
    
    print("Probabilities:")
    
    for class_name, probability in zip(
        best_gb.classes_,
        test_probabilities[i]
    ):
        print(f"  {class_name}: {probability:.2%}")

In [ ]:
from sklearn.pipeline import Pipeline

final_model = Pipeline([
    ("preprocessor", preprocessor_simple),
    ("classifier", best_gb)
])

# Train the complete pipeline on the training data
final_model.fit(X_train, y_train)

print("Final model trained successfully.")

In [ ]:
final_prediction = final_model.predict(new_user)

print("Prediction:", final_prediction[0])

In [ ]:
import joblib

# Save the complete model pipeline
joblib.dump(
    final_model,
    "ai_personal_finance_advisor_model.pkl"
)

print("Model saved successfully!")

In [ ]:
import joblib

# Save the complete model pipeline
joblib.dump(
    final_model,
    "ai_personal_finance_advisor_model.pkl"
)

print("Model saved successfully!")

In [ ]:
import joblib

# Load the saved model
loaded_model = joblib.load(
    "ai_personal_finance_advisor_model.pkl"
)

print("Saved model loaded successfully!")

In [ ]:
loaded_prediction = loaded_model.predict(new_user)

print("Prediction from saved model:")
print(loaded_prediction[0])

In [ ]:
# Expense categories that we want to analyze
expense_categories = [
    "Rent",
    "Loan_Repayment",
    "Insurance",
    "Groceries",
    "Transport",
    "Eating_Out",
    "Entertainment",
    "Utilities",
    "Healthcare",
    "Education",
    "Miscellaneous"
]

# Check that all required columns are present
missing_categories = [
    col for col in expense_categories
    if col not in data.columns
]

if missing_categories:
    print("Missing columns:", missing_categories)
else:
    print("All expense category columns are available.")

In [ ]:
# Calculate each expense category as a percentage of income

for category in expense_categories:
    ratio_column = category + "_Ratio"
    
    data[ratio_column] = (
        data[category] / data["Income"]
    )

# Display the first 5 rows
data[
    ["Income"] +
    expense_categories +
    [category + "_Ratio" for category in expense_categories]
].head()

In [ ]:
# Get the ratio columns
ratio_columns = [
    category + "_Ratio"
    for category in expense_categories
]

# Calculate the average spending ratio for each category
average_category_ratios = data[ratio_columns].mean()

# Convert decimal ratios to percentages
average_category_percentages = (
    average_category_ratios * 100
).sort_values(ascending=False)

print("Average Spending Percentage by Category:")
print()

for category_ratio, percentage in average_category_percentages.items():
    category_name = category_ratio.replace("_Ratio", "")
    print(f"{category_name}: {percentage:.2f}%")

In [ ]:
# Calculate spending ratio percentiles for each category

category_distribution = data[ratio_columns].quantile(
    [0.25, 0.50, 0.75, 0.90]
).T

# Convert ratios to percentages
category_distribution = category_distribution * 100

# Rename columns
category_distribution.columns = [
    "25th_Percentile",
    "Median",
    "75th_Percentile",
    "90th_Percentile"
]

# Remove "_Ratio" from category names
category_distribution.index = (
    category_distribution.index
    .str.replace("_Ratio", "", regex=False)
)

# Display the result
category_distribution.round(2)

In [ ]:
def analyze_expense_categories(user_data):

    results = []

    income = user_data["Income"]

    for category in expense_categories:

        # Calculate category spending as a percentage of income
        amount = user_data[category]
        ratio = (amount / income) * 100

        # Get dataset-derived reference values
        p25 = category_distribution.loc[
            category, "25th_Percentile"
        ]

        p50 = category_distribution.loc[
            category, "Median"
        ]

        p75 = category_distribution.loc[
            category, "75th_Percentile"
        ]

        p90 = category_distribution.loc[
            category, "90th_Percentile"
        ]

        # Zero spending is classified as Very Low.
        # This also handles repeated percentile values in the dataset.
        if ratio <= 0:
            status = "Very Low"
        elif ratio >= p90:
            status = "Very High"

        elif ratio >= p75:
            status = "High"

        elif ratio >= p50:
            status = "Moderate"

        elif ratio >= p25:
            status = "Low"

        else:
            status = "Very Low"

        results.append({
            "Category": category,
            "Amount": amount,
            "Income_Percentage": ratio,
            "25th_Percentile": p25,
            "Median": p50,
            "75th_Percentile": p75,
            "90th_Percentile": p90,
            "Status": status
        })

    return pd.DataFrame(results)

In [ ]:
# Select one real user from the dataset
sample_user = data.loc[0]

# Analyze the user's expense categories
sample_analysis = analyze_expense_categories(sample_user)

# Display the analysis
sample_analysis.round(2)

In [ ]:
# Select one real user from the dataset
sample_user = data.loc[0]

# Analyze the user's expense categories
sample_analysis = analyze_expense_categories(sample_user)

# Display the analysis
sample_analysis.round(2)

In [ ]:
# Calculate savings percentage of income

data["Actual_Savings_Percentage"] = (
    data["Desired_Savings"] / data["Income"]
) * 100

# Display some examples
data[
    [
        "Income",
        "Desired_Savings",
        "Desired_Savings_Percentage",
        "Actual_Savings_Percentage"
    ]
].head()

In [ ]:
# Calculate the amount of income remaining after total expenses

data["Remaining_Income"] = (
    data["Income"] - data["Total_Expenses"]
)

# Compare remaining income with desired savings
data["Savings_Feasibility"] = (
    data["Remaining_Income"] - data["Desired_Savings"]
)

# Display the first 5 records
data[
    [
        "Income",
        "Total_Expenses",
        "Remaining_Income",
        "Desired_Savings",
        "Savings_Feasibility"
    ]
].head()

In [ ]:
# Use a clearer name because the dataset represents desired savings

data["Savings_Percentage"] = (
    data["Desired_Savings"] / data["Income"]
) * 100

# Remove the misleading column created earlier
data.drop(
    columns=["Actual_Savings_Percentage"],
    inplace=True
)

# Check the result
data[
    [
        "Income",
        "Desired_Savings",
        "Savings_Percentage",
        "Remaining_Income",
        "Savings_Feasibility"
    ]
].head()

In [ ]:
data["Savings_Feasibility"].describe()

In [ ]:
print(
    "Negative feasibility:",
    (data["Savings_Feasibility"] < 0).sum()
)

print(
    "Zero feasibility:",
    (data["Savings_Feasibility"] == 0).sum()
)

print(
    "Positive feasibility:",
    (data["Savings_Feasibility"] > 0).sum()
)

In [ ]:
data["Savings_Buffer_Percentage"] = (
    data["Savings_Feasibility"] / data["Income"]
) * 100

In [ ]:
data["Savings_Buffer_Percentage"].describe()

In [ ]:
data[
    data["Savings_Buffer_Percentage"] < 0
][
    [
        "Income",
        "Total_Expenses",
        "Desired_Savings",
        "Savings_Feasibility",
        "Savings_Buffer_Percentage"
    ]
].head(10)

In [ ]:
data["Savings_Feasibility_Clean"] = (
    data["Savings_Feasibility"].round(2)
)

In [ ]:
data[
    [
        "Savings_Feasibility",
        "Savings_Feasibility_Clean"
    ]
].head(10)

In [ ]:
print(
    "Negative:",
    (data["Savings_Feasibility_Clean"] < 0).sum()
)

print(
    "Zero:",
    (data["Savings_Feasibility_Clean"] == 0).sum()
)

print(
    "Positive:",
    (data["Savings_Feasibility_Clean"] > 0).sum()
)

In [ ]:
def get_savings_status(row):
    
    if row["Savings_Feasibility_Clean"] < 0:
        return "Savings Goal Not Feasible"
    
    elif row["Savings_Feasibility_Clean"] == 0:
        return "Savings Goal Exactly Feasible"
    
    else:
        return "Savings Goal Feasible"


data["Savings_Status"] = data.apply(
    get_savings_status,
    axis=1
)

In [ ]:
data["Savings_Status"].value_counts()

In [ ]:
buffer_q25 = data["Savings_Buffer_Percentage"].quantile(0.25)
buffer_q50 = data["Savings_Buffer_Percentage"].quantile(0.50)
buffer_q75 = data["Savings_Buffer_Percentage"].quantile(0.75)

print("25th percentile:", buffer_q25)
print("50th percentile:", buffer_q50)
print("75th percentile:", buffer_q75)

In [ ]:
def get_detailed_savings_status(row):
    
    buffer = row["Savings_Buffer_Percentage"]
    
    if buffer < 0:
        return "Savings Goal Not Feasible"
    
    elif buffer < buffer_q25:
        return "Feasible - Low Buffer"
    
    elif buffer < buffer_q50:
        return "Feasible - Moderate Buffer"
    
    elif buffer < buffer_q75:
        return "Feasible - Good Buffer"
    
    else:
        return "Feasible - Strong Buffer"


data["Detailed_Savings_Status"] = data.apply(
    get_detailed_savings_status,
    axis=1
)

In [ ]:
data["Detailed_Savings_Status"].value_counts()

In [ ]:
def get_savings_recommendation(row):
    
    status = row["Detailed_Savings_Status"]
    
    if status == "Savings Goal Not Feasible":
        return (
            "Your desired savings goal is currently not feasible "
            "after your listed expenses. Consider reducing expenses "
            "or lowering your desired savings target."
        )
    
    elif status == "Feasible - Low Buffer":
        return (
            "Your savings goal is feasible, but your remaining "
            "financial buffer is relatively low. Consider keeping "
            "some additional amount available for unexpected expenses."
        )
    
    elif status == "Feasible - Moderate Buffer":
        return (
            "Your savings goal is feasible with a moderate remaining "
            "buffer. Continue monitoring your expenses and savings."
        )
    
    elif status == "Feasible - Good Buffer":
        return (
            "Your savings goal is feasible with a good remaining "
            "buffer. Continue maintaining your current financial balance."
        )
    
    else:
        return (
            "Your savings goal is feasible with a strong remaining "
            "buffer. You have relatively more flexibility after "
            "meeting your desired savings goal."
        )


data["Savings_Recommendation"] = data.apply(
    get_savings_recommendation,
    axis=1
)

In [ ]:
data[
    [
        "Income",
        "Desired_Savings",
        "Savings_Feasibility_Clean",
        "Savings_Buffer_Percentage",
        "Detailed_Savings_Status",
        "Savings_Recommendation"
    ]
].head(10)

In [ ]:
data["loan_payment_ratio"].describe()

In [ ]:
print(
    "No Loan Repayment:",
    (data["Loan_Repayment"] == 0).sum()
)

print(
    "With Loan Repayment:",
    (data["Loan_Repayment"] > 0).sum()
)

In [ ]:
debt_q75 = data["loan_payment_ratio"].quantile(0.75)
debt_q90 = data["loan_payment_ratio"].quantile(0.90)

print("75th percentile:", debt_q75)
print("90th percentile:", debt_q90)

In [ ]:
def get_debt_status(row):
    
    ratio = row["loan_payment_ratio"]
    
    if ratio == 0:
        return "No Loan Repayment"
    
    elif ratio < debt_q75:
        return "Normal Loan Burden"
    
    elif ratio < debt_q90:
        return "High Loan Burden"
    
    else:
        return "Very High Loan Burden"


data["Debt_Status"] = data.apply(
    get_debt_status,
    axis=1
)

In [ ]:
data["Debt_Status"].value_counts()

In [ ]:
def get_debt_recommendation(row):

    status = row["Debt_Status"]

    if status == "No Loan Repayment":
        return (
            "You currently have no loan repayment burden "
            "based on the provided data."
        )

    elif status == "Normal Loan Burden":
        return (
            "Your loan repayment is relatively moderate compared "
            "with your income. Continue monitoring your EMI and "
            "avoid taking unnecessary additional debt."
        )

    elif status == "High Loan Burden":
        return (
            "Your loan repayment takes a relatively high share "
            "of your income. Consider controlling additional debt "
            "and reviewing your monthly expenses."
        )

    else:
        return (
            "Your loan repayment takes a very high share of your "
            "income. Consider reducing additional borrowing and "
            "reviewing your repayment strategy carefully."
        )


data["Debt_Recommendation"] = data.apply(
    get_debt_recommendation,
    axis=1
)

In [ ]:
data[
    [
        "Income",
        "Loan_Repayment",
        "loan_payment_ratio",
        "Debt_Status",
        "Debt_Recommendation"
    ]
].head(10)

In [ ]:
print(data.columns.tolist())

In [ ]:
status_columns = [
    col for col in data.columns
    if "Status" in col or "Recommendation" in col
]

print(status_columns)

In [ ]:
expense_status = {}

for category in expense_categories:

    ratio_column = category + "_Ratio"

    q25 = data[ratio_column].quantile(0.25)
    q50 = data[ratio_column].quantile(0.50)
    q75 = data[ratio_column].quantile(0.75)
    q90 = data[ratio_column].quantile(0.90)

    def classify_expense(value):

        # Keep zero spending at the lowest level even when
        # multiple dataset percentiles are equal to zero.
        if value <= 0:
            return "Very Low"

        elif value >= q90:
            return "Very High"

        elif value >= q75:
            return "High"

        elif value >= q50:
            return "Moderate"

        elif value >= q25:
            return "Low"

        else:
            return "Very Low"

    status_column = category + "_Status"

    data[status_column] = data[ratio_column].apply(
        classify_expense
    )

    expense_status[category] = status_column

In [ ]:
print(expense_status)

In [ ]:
data[
    [
        "Rent_Status",
        "Loan_Repayment_Status",
        "Insurance_Status",
        "Groceries_Status",
        "Transport_Status",
        "Eating_Out_Status",
        "Entertainment_Status",
        "Utilities_Status",
        "Healthcare_Status",
        "Education_Status",
        "Miscellaneous_Status"
    ]
].head()

## expence_recomendation

In [ ]:
def get_expense_recommendation(row, category):

    status = row[category + "_Status"]
    category_name = category.replace("_", " ").lower()

    if status == "Very High":
        return (
            f"Your {category_name} spending is relatively very high "
            "compared with the dataset. Consider reviewing and "
            "reducing this expense where possible."
        )

    elif status == "High":
        return (
            f"Your {category_name} spending is relatively high "
            "compared with the dataset. Consider monitoring or "
            "reducing this expense where possible."
        )

    elif status == "Moderate":
        return (
            f"Your {category_name} spending is around the middle-to-upper "
            "range of the dataset. Monitor it and look for possible savings."
        )

    elif status == "Low":
        return (
            f"Your {category_name} spending is relatively low "
            "compared with the dataset."
        )

    else:
        return (
            f"Your {category_name} spending is very low "
            "compared with the dataset. Check that essential needs "
            "are still being covered."
        )

In [ ]:
for category in expense_categories:
    
    recommendation_column = category + "_Recommendation"
    
    data[recommendation_column] = data.apply(
        lambda row: get_expense_recommendation(
            row,
            category
        ),
        axis=1
    )

In [ ]:
data[
    [
        "Groceries_Status",
        "Groceries_Recommendation",
        "Insurance_Status",
        "Insurance_Recommendation",
        "Entertainment_Status",
        "Entertainment_Recommendation"
    ]
].head(10)

In [ ]:
def generate_financial_recommendations(row):

    recommendations = []

    # -----------------------------------
    # 1. EXPENSE RECOMMENDATIONS
    # -----------------------------------

    for category in expense_categories:

        status = row[category + "_Status"]
        category_name = category.replace("_", " ").lower()

        if status == "Very High":
            recommendations.append(
                f"High priority: Review your {category_name} "
                f"spending because it is relatively very high."
            )

        elif status == "High":
            recommendations.append(
                f"Consider reviewing your {category_name} "
                f"spending because it is relatively high."
            )

        elif status == "Moderate":
            recommendations.append(
                f"Monitor your {category_name} spending "
                f"and look for opportunities to save."
            )

    # -----------------------------------
    # 2. DEBT RECOMMENDATION
    # -----------------------------------

    debt_status = row["Debt_Status"]

    if debt_status == "Very High Loan Burden":

        recommendations.append(
            "Your loan repayment takes a relatively very "
            "high share of your income. Review your debt "
            "and avoid unnecessary additional borrowing."
        )

    elif debt_status == "High Loan Burden":

        recommendations.append(
            "Your loan repayment takes a relatively high "
            "share of your income. Monitor your debt burden "
            "and control additional borrowing."
        )

    # -----------------------------------
    # 3. SAVINGS RECOMMENDATION
    # -----------------------------------

    savings_status = row["Savings_Status"]

    if savings_status == "Savings Goal Not Feasible":

        recommendations.append(
            "Your current savings goal is not feasible "
            "with your present income and expenses. "
            "Consider reducing expenses or adjusting "
            "your savings target."
        )

    elif savings_status == "Savings Goal Exactly Feasible":

        recommendations.append(
            "Your savings goal is achievable, but you "
            "have very little financial buffer. Monitor "
            "your expenses carefully."
        )

    # -----------------------------------
    # 4. FINANCIAL HEALTH
    # -----------------------------------

    health = row["Financial_Health"]

    if health == "Needs Improvement":

        recommendations.append(
            "Your overall financial health needs attention. "
            "Focus on controlling expenses, managing debt, "
            "and maintaining a realistic savings goal."
        )

    elif health == "Moderate":

        recommendations.append(
            "Your overall financial health is moderate. "
            "Review the highlighted expense and savings "
            "areas to improve your financial position."
        )

    elif health == "Healthy":

        recommendations.append(
            "Your overall financial health is healthy. "
            "Continue monitoring expenses, debt, and savings "
            "to maintain your current position."
        )

    return recommendations

In [ ]:
data["Final_Recommendations"] = data.apply(
    generate_financial_recommendations,
    axis=1
)

In [ ]:
print("Financial Health:")
print(data.loc[0, "Financial_Health"])

print("\nSavings Status:")
print(data.loc[0, "Savings_Status"])

print("\nDebt Status:")
print(data.loc[0, "Debt_Status"])

print("\nRecommendations:")
for recommendation in data.loc[0, "Final_Recommendations"]:
    print("-", recommendation)

In [ ]:
user_1 = pd.DataFrame([{
    "Income": 100000,
    "Age": 35,
    "Dependents": 1,
    "City_Tier": "Tier_1",
    "Rent": 20000,
    "Loan_Repayment": 2000,
    "Insurance": 3000,
    "Groceries": 8000,
    "Transport": 5000,
    "Eating_Out": 2000,
    "Entertainment": 1500,
    "Utilities": 4000,
    "Healthcare": 3000,
    "Education": 3000,
    "Miscellaneous": 1500,
    "Desired_Savings_Percentage": 20,
    "Desired_Savings": 20000
}])

In [ ]:
user_2 = pd.DataFrame([{
    "Income": 50000,
    "Age": 30,
    "Dependents": 1,
    "City_Tier": "Tier_2",
    "Rent": 10000,
    "Loan_Repayment": 4000,
    "Insurance": 1500,
    "Groceries": 6000,
    "Transport": 3000,
    "Eating_Out": 7000,
    "Entertainment": 6000,
    "Utilities": 2500,
    "Healthcare": 2000,
    "Education": 1500,
    "Miscellaneous": 2000,
    "Desired_Savings_Percentage": 15,
    "Desired_Savings": 7500
}])

In [ ]:
user_3 = pd.DataFrame([{
    "Income": 30000,
    "Age": 28,
    "Dependents": 3,
    "City_Tier": "Tier_2",
    "Rent": 12000,
    "Loan_Repayment": 8000,
    "Insurance": 1500,
    "Groceries": 6000,
    "Transport": 3000,
    "Eating_Out": 2000,
    "Entertainment": 1500,
    "Utilities": 2500,
    "Healthcare": 1500,
    "Education": 1000,
    "Miscellaneous": 1500,
    "Desired_Savings_Percentage": 15,
    "Desired_Savings": 4500
}])

In [ ]:
def analyze_user_finances(
    income,
    age,
    dependents,
    city_tier,
    rent,
    loan_repayment,
    insurance,
    groceries,
    transport,
    eating_out,
    entertainment,
    utilities,
    healthcare,
    education,
    miscellaneous,
    desired_savings
):

    # =========================================================
    # 1. CREATE USER DATAFRAME
    # =========================================================

    user = pd.DataFrame([{
        "Income": income,
        "Age": age,
        "Dependents": dependents,
        "City_Tier": city_tier,
        "Rent": rent,
        "Loan_Repayment": loan_repayment,
        "Insurance": insurance,
        "Groceries": groceries,
        "Transport": transport,
        "Eating_Out": eating_out,
        "Entertainment": entertainment,
        "Utilities": utilities,
        "Healthcare": healthcare,
        "Education": education,
        "Miscellaneous": miscellaneous,
        "Desired_Savings": desired_savings
    }])


    # =========================================================
    # 2. EXPENSE COLUMNS
    # =========================================================

    expense_columns = [
        "Rent",
        "Loan_Repayment",
        "Insurance",
        "Groceries",
        "Transport",
        "Eating_Out",
        "Entertainment",
        "Utilities",
        "Healthcare",
        "Education",
        "Miscellaneous"
    ]


    # =========================================================
    # 3. TOTAL EXPENSES
    # =========================================================

    user["Total_Expenses"] = (
        user[expense_columns].sum(axis=1)
    )


    # =========================================================
    # 4. REMAINING INCOME
    # =========================================================

    user["Remaining_Income"] = (
        user["Income"]
        - user["Total_Expenses"]
    )


    # =========================================================
    # 5. EXPENSE RATIOS
    # =========================================================

    for category in expense_columns:

        ratio_column = category + "_Ratio"

        user[ratio_column] = (
            user[category] / user["Income"]
        )


    # =========================================================
    # 6. EXPENSE THRESHOLDS
    # =========================================================

    # Use thresholds calculated from the original dataset.
    # category_distribution contains percentages for each category.
    expense_thresholds = {}

    for category in expense_columns:

        expense_thresholds[category] = {
            "very_low": category_distribution.loc[
                category, "25th_Percentile"
            ],
            "low": category_distribution.loc[
                category, "Median"
            ],
            "moderate": category_distribution.loc[
                category, "75th_Percentile"
            ],
            "high": category_distribution.loc[
                category, "90th_Percentile"
            ]
        }


    # =========================================================
    # 7. EXPENSE STATUS
    # =========================================================

    for category in expense_columns:

        ratio_column = category + "_Ratio"
        status_column = category + "_Status"

        percentage = (
            user[ratio_column] * 100
        )

        very_low_threshold = (
            expense_thresholds[category]["very_low"]
        )

        low_threshold = (
            expense_thresholds[category]["low"]
        )

        moderate_threshold = (
            expense_thresholds[category]["moderate"]
        )

        high_threshold = (
            expense_thresholds[category]["high"]
        )

        # Start with the lowest level
        user[status_column] = "Very Low"

        # Low
        user.loc[
            percentage >= very_low_threshold,
            status_column
        ] = "Low"

        # Moderate
        user.loc[
            percentage >= low_threshold,
            status_column
        ] = "Moderate"

        # High
        user.loc[
            percentage >= moderate_threshold,
            status_column
        ] = "High"

        # Very High
        user.loc[
            percentage >= high_threshold,
            status_column
        ] = "Very High"

        # Zero expense should remain Very Low, even if
        # some dataset percentile thresholds are also zero.
        user.loc[
            percentage <= 0,
            status_column
        ] = "Very Low"


    
    # =========================================================
    # 8. EXPENSE RECOMMENDATIONS
    # =========================================================

    recommendations = []

    for category in expense_columns:

        status_column = category + "_Status"

        status = (
            user[status_column].iloc[0]
        )

        category_name = (
            category.replace("_", " ").lower()
        )

        if status == "Very High":

            recommendations.append(
                f"High priority: Review your "
                f"{category_name} spending because "
                f"it is relatively very high."
            )

        elif status == "High":

            recommendations.append(
                f"Consider reviewing your "
                f"{category_name} spending because "
                f"it is relatively high."
            )

        elif status == "Moderate":

            recommendations.append(
                f"Monitor your {category_name} spending "
                f"and look for opportunities to save."
            )

        elif status == "Low":

            recommendations.append(
                f"Your {category_name} spending is "
                f"relatively low compared with the dataset."
            )

        else:

            recommendations.append(
                f"Your {category_name} spending is "
                f"very low compared with the dataset."
            )

    # IMPORTANT: Save recommendations in the DataFrame
    user["Expense_Recommendations"] = [
        recommendations
    ]


    # =========================================================
    # 9. SAVINGS FEASIBILITY
    # =========================================================

    user["Savings_Feasibility"] = (
        user["Remaining_Income"]
        - user["Desired_Savings"]
    )


    # =========================================================
    # 10. SAVINGS STATUS
    # =========================================================

    user["Savings_Status"] = (
        "Savings Goal Feasible"
    )

    user.loc[
        user["Savings_Feasibility"] < 0,
        "Savings_Status"
    ] = "Savings Goal Not Feasible"

    user.loc[
        user["Savings_Feasibility"] == 0,
        "Savings_Status"
    ] = "Savings Goal Exactly Feasible"


    # =========================================================
    # 11. SAVINGS BUFFER
    # =========================================================

    user["Savings_Buffer_Percentage"] = (
        user["Savings_Feasibility"]
        / user["Income"]
    ) * 100


    # =========================================================
    # 12. DETAILED SAVINGS STATUS
    # =========================================================

    user["Detailed_Savings_Status"] = (
        "Feasible - Low Buffer"
    )

    user.loc[
        user["Savings_Feasibility"] < 0,
        "Detailed_Savings_Status"
    ] = "Savings Goal Not Feasible"


    user.loc[
        (
            (user["Savings_Feasibility"] >= 0)
            &
            (user["Savings_Buffer_Percentage"] < 8.656757)
        ),
        "Detailed_Savings_Status"
    ] = "Feasible - Low Buffer"


    user.loc[
        (
            (user["Savings_Buffer_Percentage"] >= 8.656757)
            &
            (user["Savings_Buffer_Percentage"] < 16.373592)
        ),
        "Detailed_Savings_Status"
    ] = "Feasible - Moderate Buffer"


    user.loc[
        (
            (user["Savings_Buffer_Percentage"] >= 16.373592)
            &
            (user["Savings_Buffer_Percentage"] < 23.513960)
        ),
        "Detailed_Savings_Status"
    ] = "Feasible - Good Buffer"


    user.loc[
        user["Savings_Buffer_Percentage"] >= 23.513960,
        "Detailed_Savings_Status"
    ] = "Feasible - Strong Buffer"


    # =========================================================
    # 13. SAVINGS RECOMMENDATION
    # =========================================================

    savings_status = (
        user["Detailed_Savings_Status"].iloc[0]
    )

    if savings_status == "Savings Goal Not Feasible":

        savings_recommendation = (
            "Your desired savings goal is currently "
            "not feasible. Consider reducing expenses "
            "or adjusting your savings target."
        )

    elif savings_status == "Feasible - Low Buffer":

        savings_recommendation = (
            "Your savings goal is feasible, but your "
            "remaining financial buffer is low. "
            "Consider reducing unnecessary expenses "
            "to create more financial flexibility."
        )

    elif savings_status == "Feasible - Moderate Buffer":

        savings_recommendation = (
            "Your savings goal is feasible with a "
            "moderate financial buffer. Continue "
            "monitoring your expenses and savings."
        )

    elif savings_status == "Feasible - Good Buffer":

        savings_recommendation = (
            "Your savings goal is feasible with a "
            "good financial buffer. Your current "
            "savings position provides reasonable "
            "flexibility."
        )

    else:

        savings_recommendation = (
            "Your savings goal is feasible with a "
            "strong financial buffer. You have good "
            "room after meeting your savings goal."
        )

    user["Savings_Recommendation"] = (
        savings_recommendation
    )


    # =========================================================
    # 14. LOAN PAYMENT RATIO
    # =========================================================

    user["loan_payment_ratio"] = (
        user["Loan_Repayment"]
        / user["Income"]
    )


    # =========================================================
    # 15. DEBT STATUS
    # =========================================================

    user["Debt_Status"] = (
        "No Loan Repayment"
    )


    # Normal Loan Burden
    user.loc[
        (
            (user["loan_payment_ratio"] > 0)
            &
            (user["loan_payment_ratio"] < 0.106328)
        ),
        "Debt_Status"
    ] = "Normal Loan Burden"


    # High Loan Burden
    user.loc[
        (
            (user["loan_payment_ratio"] >= 0.106328)
            &
            (user["loan_payment_ratio"] < 0.162882)
        ),
        "Debt_Status"
    ] = "High Loan Burden"


    # Very High Loan Burden
    user.loc[
        user["loan_payment_ratio"] >= 0.162882,
        "Debt_Status"
    ] = "Very High Loan Burden"


    # =========================================================
    # 16. DEBT RECOMMENDATION
    # =========================================================

    debt_status = (
        user["Debt_Status"].iloc[0]
    )

    if debt_status == "No Loan Repayment":

        debt_recommendation = (
            "You currently have no loan repayment "
            "burden based on the provided information."
        )

    elif debt_status == "Normal Loan Burden":

        debt_recommendation = (
            "Your loan repayment is relatively moderate "
            "compared with the income distribution in "
            "the dataset. Continue monitoring your "
            "loan burden."
        )

    elif debt_status == "High Loan Burden":

        debt_recommendation = (
            "Your loan repayment takes a relatively "
            "high share of your income. Consider "
            "reviewing your loan repayment obligations "
            "and monthly budget."
        )

    else:

        debt_recommendation = (
            "Your loan repayment takes a very high "
            "share of your income. Reducing debt "
            "burden and reviewing your repayment plan "
            "may improve your financial flexibility."
        )

    user["Debt_Recommendation"] = (
        debt_recommendation
    )


    # =========================================================
    # 17. FINANCIAL HEALTH MODEL INPUT
    # =========================================================
    
    new_user_model_input = user[
        [
            "Age",
            "Dependents",
            "City_Tier",
            "Income",
            "Total_Expenses",
            "Loan_Repayment",
            "Desired_Savings"
        ]
    ]


    # =========================================================
    # 18. ENCODE USER INPUT
    # =========================================================

    new_user_encoded = (
        preprocessor_simple.transform(
            new_user_model_input
        )
    )


    # =========================================================
    # 19. FINANCIAL HEALTH PREDICTION
    # =========================================================

    financial_health_prediction = (
        best_gb.predict(
            new_user_encoded
        )
    )

    user["Financial_Health"] = (
        financial_health_prediction[0]
    )


    # =========================================================
    # 20. FINANCIAL HEALTH PROBABILITIES
    # =========================================================

    financial_health_probability = (
        best_gb.predict_proba(
            new_user_encoded
        )
    )

    probability_dict = {}

    for class_name, probability in zip(
        best_gb.classes_,
        financial_health_probability[0]
    ):

        probability_dict[class_name] = (
            probability
        )

    user["Financial_Health_Probabilities"] = [
        probability_dict
    ]


    # =========================================================
    # 21. OVERALL RECOMMENDATIONS
    # =========================================================

    overall_recommendations = []

    # Expense recommendations
    for recommendation in recommendations:

        overall_recommendations.append(
            recommendation
        )


    # Savings recommendation
    overall_recommendations.append(
        savings_recommendation
    )


    # Debt recommendation
    overall_recommendations.append(
        debt_recommendation
    )


    # Financial health recommendation
    financial_health = (
        financial_health_prediction[0]
    )

    if financial_health == "Healthy":

        overall_recommendations.append(
            "Your overall financial health is healthy. "
            "Continue maintaining your current financial "
            "habits while monitoring your savings and "
            "expenses."
        )

    elif financial_health == "Moderate":

        overall_recommendations.append(
            "Your overall financial health is moderate. "
            "Review the highlighted expense, savings, "
            "and debt areas to improve your financial "
            "position."
        )

    else:

        overall_recommendations.append(
            "Your overall financial health needs "
            "improvement. Focus on controlling "
            "unnecessary expenses, managing debt, "
            "and maintaining a realistic savings goal."
        )


    user["Overall_Recommendations"] = [
        overall_recommendations
    ]


    # =========================================================
    # 22. RETURN COMPLETE USER ANALYSIS
    # =========================================================

    return user

In [ ]:
test_user = analyze_user_finances(
    income=50000,
    age=30,
    dependents=1,
    city_tier="Tier_2",
    rent=10000,
    loan_repayment=4000,
    insurance=1500,
    groceries=6000,
    transport=3000,
    eating_out=3000,
    entertainment=2000,
    utilities=2500,
    healthcare=2000,
    education=1500,
    miscellaneous=1000,
    desired_savings=7500
)

print("Analysis completed successfully.")

In [ ]:
status_columns = [
    "Rent_Status",
    "Loan_Repayment_Status",
    "Insurance_Status",
    "Groceries_Status",
    "Transport_Status",
    "Eating_Out_Status",
    "Entertainment_Status",
    "Utilities_Status",
    "Healthcare_Status",
    "Education_Status",
    "Miscellaneous_Status"
]

print(test_user[status_columns].T)

In [ ]:
model_features = [
    "Age",
    "Dependents",
    "City_Tier",
    "Income",
    "Total_Expenses",
    "Loan_Repayment",
    "Desired_Savings"
]

new_user_model_input = test_user[model_features]

print(new_user_model_input)

In [ ]:
new_user_encoded = preprocessor_simple.transform(
    new_user_model_input
)

print("Encoded shape:", new_user_encoded.shape)

In [ ]:
financial_health_prediction = best_gb.predict(
    new_user_encoded
)

print("Predicted Financial Health:")
print(financial_health_prediction[0])

In [ ]:
financial_health_probability = best_gb.predict_proba(
    new_user_encoded
)

print("\nPrediction Probabilities:")

for class_name, probability in zip(
    best_gb.classes_,
    financial_health_probability[0]
):
    print(f"{class_name}: {probability:.2%}")

In [ ]:
# Expense thresholds derived from the dataset percentiles
expense_thresholds = {}

for category in expense_categories:

    expense_thresholds[category] = {
        "very_low": category_distribution.loc[
            category, "25th_Percentile"
        ],
        "low": category_distribution.loc[
            category, "Median"
        ],
        "moderate": category_distribution.loc[
            category, "75th_Percentile"
        ],
        "high": category_distribution.loc[
            category, "90th_Percentile"
        ]
    }

# Display the thresholds as percentages
pd.DataFrame(expense_thresholds).T.round(2)

In [ ]:
print(test_user.columns.tolist())

In [ ]:
print(test_user.columns.tolist())

In [ ]:
for recommendation in test_user["Expense_Recommendations"].iloc[0]:

    print("-", recommendation)

In [ ]:
print(
    test_user[
        [
            "Financial_Health",
            "Total_Expenses",
            "Remaining_Income",
            "Desired_Savings",
            "Savings_Feasibility",
            "Savings_Buffer_Percentage",
            "Savings_Status",
            "Detailed_Savings_Status",
            "Debt_Status"
        ]
    ].T
)

In [ ]:
print("\nExpense Recommendations:")

for recommendation in test_user[
    "Expense_Recommendations"
].iloc[0]:

    print("-", recommendation)

In [ ]:
print("\nOverall Recommendations:")

for recommendation in test_user[
    "Overall_Recommendations"
].iloc[0]:

    print("-", recommendation)

In [ ]:
print("\nFinancial Health Probabilities:")

for class_name, probability in (
    test_user[
        "Financial_Health_Probabilities"
    ].iloc[0].items()
):

    print(
        f"{class_name}: {probability:.2%}"
    )

In [ ]:
# ============================================================
# FINAL TESTING OF AI PERSONAL FINANCE ADVISOR
# ============================================================

test_users = [

    # --------------------------------------------------------
    # USER 1: Strong financial position
    # --------------------------------------------------------
    {
        "name": "User 1 - Strong Financial Position",

        "income": 100000,
        "age": 30,
        "dependents": 1,
        "city_tier": "Tier_1",

        "rent": 20000,
        "loan_repayment": 5000,
        "insurance": 3000,
        "groceries": 10000,
        "transport": 5000,
        "eating_out": 2000,
        "entertainment": 2000,
        "utilities": 5000,
        "healthcare": 3000,
        "education": 5000,
        "miscellaneous": 2000,

        "desired_savings": 30000
    },


    # --------------------------------------------------------
    # USER 2: Moderate financial position
    # --------------------------------------------------------
    {
        "name": "User 2 - Moderate Financial Position",

        "income": 50000,
        "age": 30,
        "dependents": 1,
        "city_tier": "Tier_2",

        "rent": 10000,
        "loan_repayment": 4000,
        "insurance": 1500,
        "groceries": 6000,
        "transport": 3000,
        "eating_out": 3000,
        "entertainment": 2000,
        "utilities": 2500,
        "healthcare": 2000,
        "education": 1500,
        "miscellaneous": 1000,

        "desired_savings": 7500
    },


    # --------------------------------------------------------
    # USER 3: Financially stressed
    # --------------------------------------------------------
    {
        "name": "User 3 - Financially Stressed",

        "income": 30000,
        "age": 28,
        "dependents": 3,
        "city_tier": "Tier_2",

        "rent": 10000,
        "loan_repayment": 7000,
        "insurance": 2000,
        "groceries": 6000,
        "transport": 3000,
        "eating_out": 2000,
        "entertainment": 2000,
        "utilities": 2500,
        "healthcare": 1500,
        "education": 2000,
        "miscellaneous": 1500,

        "desired_savings": 5000
    },


    # --------------------------------------------------------
    # USER 4: High discretionary spending
    # --------------------------------------------------------
    {
        "name": "User 4 - High Discretionary Spending",

        "income": 60000,
        "age": 25,
        "dependents": 0,
        "city_tier": "Tier_2",

        "rent": 12000,
        "loan_repayment": 3000,
        "insurance": 1500,
        "groceries": 7000,
        "transport": 3000,
        "eating_out": 5000,
        "entertainment": 5000,
        "utilities": 2500,
        "healthcare": 1500,
        "education": 1000,
        "miscellaneous": 3000,

        "desired_savings": 10000
    },


    # --------------------------------------------------------
    # USER 5: High debt burden
    # --------------------------------------------------------
    {
        "name": "User 5 - High Debt Burden",

        "income": 50000,
        "age": 35,
        "dependents": 2,
        "city_tier": "Tier_2",

        "rent": 10000,
        "loan_repayment": 10000,
        "insurance": 2000,
        "groceries": 6000,
        "transport": 3000,
        "eating_out": 1500,
        "entertainment": 1000,
        "utilities": 2500,
        "healthcare": 2000,
        "education": 1500,
        "miscellaneous": 1000,

        "desired_savings": 5000
    }
]


# ============================================================
# RUN ALL TEST USERS
# ============================================================

all_test_results = []


for test in test_users:

    result = analyze_user_finances(

        income=test["income"],
        age=test["age"],
        dependents=test["dependents"],
        city_tier=test["city_tier"],

        rent=test["rent"],
        loan_repayment=test["loan_repayment"],
        insurance=test["insurance"],
        groceries=test["groceries"],
        transport=test["transport"],
        eating_out=test["eating_out"],
        entertainment=test["entertainment"],
        utilities=test["utilities"],
        healthcare=test["healthcare"],
        education=test["education"],
        miscellaneous=test["miscellaneous"],

        desired_savings=test["desired_savings"]
    )

    all_test_results.append({
        "User": test["name"],

        "Income": test["income"],

        "Total_Expenses": result[
            "Total_Expenses"
        ].iloc[0],

        "Remaining_Income": result[
            "Remaining_Income"
        ].iloc[0],

        "Financial_Health": result[
            "Financial_Health"
        ].iloc[0],

        "Savings_Status": result[
            "Savings_Status"
        ].iloc[0],

        "Detailed_Savings_Status": result[
            "Detailed_Savings_Status"
        ].iloc[0],

        "Debt_Status": result[
            "Debt_Status"
        ].iloc[0]
    })


# ============================================================
# CREATE FINAL TEST SUMMARY
# ============================================================

final_test_results = pd.DataFrame(
    all_test_results
)


print("\n")
print("=" * 100)
print("              FINAL AI PERSONAL FINANCE ADVISOR TEST")
print("=" * 100)

print(
    final_test_results.to_string(
        index=False
    )
)

print("=" * 100)

In [ ]:
print(
    "\nNumber of test users:",
    len(final_test_results)
)

print(
    "Number of completed results:",
    len(final_test_results.dropna())
)

In [ ]:
print("\nFinancial Health Distribution:")

print(
    final_test_results[
        "Financial_Health"
    ].value_counts()
)

In [ ]:
print("\nSavings Status Distribution:")

print(
    final_test_results[
        "Savings_Status"
    ].value_counts()
)

## Save the trained model and preprocessor

In [ ]:
import joblib
import os


# ============================================================
# CREATE MODEL DIRECTORY
# ============================================================

model_dir = "saved_model"

os.makedirs(
    model_dir,
    exist_ok=True
)


# ============================================================
# SAVE GRADIENT BOOSTING MODEL
# ============================================================

model_path = os.path.join(
    model_dir,
    "financial_health_model.pkl"
)

joblib.dump(
    best_gb,
    model_path
)


# ============================================================
# SAVE PREPROCESSOR
# ============================================================

preprocessor_path = os.path.join(
    model_dir,
    "financial_health_preprocessor.pkl"
)

joblib.dump(
    preprocessor_simple,
    preprocessor_path
)


# ============================================================
# CHECK FILES
# ============================================================

print("Model saved successfully.")
print("Model path:", model_path)
print("Model exists:", os.path.exists(model_path))

print()

print("Preprocessor saved successfully.")
print("Preprocessor path:", preprocessor_path)
print(
    "Preprocessor exists:",
    os.path.exists(preprocessor_path)
)

## load model

In [ ]:
import joblib


# ============================================================
# LOAD SAVED MODEL
# ============================================================

loaded_model = joblib.load(
    "saved_model/financial_health_model.pkl"
)


# ============================================================
# LOAD SAVED PREPROCESSOR
# ============================================================

loaded_preprocessor = joblib.load(
    "saved_model/financial_health_preprocessor.pkl"
)


print("Saved model loaded successfully.")
print("Saved preprocessor loaded successfully.")

## Test the loaded model

In [ ]:
loaded_user_input = pd.DataFrame([{
    "Age": 30,
    "Dependents": 1,
    "City_Tier": "Tier_2",
    "Income": 50000,
    "Total_Expenses": 36500,
    "Loan_Repayment": 4000,
    "Desired_Savings": 7500
}])


loaded_user_encoded = (
    loaded_preprocessor.transform(
        loaded_user_input
    )
)


loaded_prediction = (
    loaded_model.predict(
        loaded_user_encoded
    )
)


loaded_probability = (
    loaded_model.predict_proba(
        loaded_user_encoded
    )
)


print("Prediction from saved system:")
print(
    loaded_prediction[0]
)


print("\nProbabilities:")

for class_name, probability in zip(
    loaded_model.classes_,
    loaded_probability[0]
):

    print(
        f"{class_name}: {probability:.2%}"
    )

In [ ]:
import os


print("Model file:")
print(
    "Exists:",
    os.path.exists(
        "saved_model/financial_health_model.pkl"
    )
)


print("\nPreprocessor file:")
print(
    "Exists:",
    os.path.exists(
        "saved_model/financial_health_preprocessor.pkl"
    )
)


print("\nSaved model folder contents:")

print(
    os.listdir("saved_model")
)

## balanced

In [ ]:
# ============================================================
# BALANCED GRADIENT BOOSTING MODEL
# ============================================================

from sklearn.utils.class_weight import compute_sample_weight

# Calculate balanced sample weights for the training data
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

print("Sample weights created successfully.")
print("Number of training samples:", len(sample_weights))

In [ ]:
# ============================================================
# CREATE BALANCED GRADIENT BOOSTING MODEL
# ============================================================

from sklearn.ensemble import GradientBoostingClassifier

balanced_gb = GradientBoostingClassifier(
    learning_rate=0.1,
    max_depth=3,
    min_samples_leaf=2,
    min_samples_split=5,
    n_estimators=200,
    random_state=42
)

print("Balanced Gradient Boosting model created.")

In [ ]:
# ============================================================
# TRAIN BALANCED MODEL
# ============================================================

balanced_gb.fit(
    X_train_encoded,
    y_train,
    sample_weight=sample_weights
)

print("Balanced Gradient Boosting training completed.")

In [ ]:
   # ============================================================
# PREDICTION ON ORIGINAL TEST DATA
# ============================================================

y_pred_balanced = balanced_gb.predict(X_test_encoded)

print("Prediction completed.")
print("Number of test predictions:", len(y_pred_balanced))

In [ ]:
# ============================================================
# BALANCED MODEL EVALUATION
# ============================================================

from sklearn.metrics import accuracy_score, f1_score

balanced_accuracy = accuracy_score(
    y_test,
    y_pred_balanced
)

balanced_macro_f1 = f1_score(
    y_test,
    y_pred_balanced,
    average="macro"
)

print("Balanced Gradient Boosting Results")
print("----------------------------------")
print("Accuracy :", balanced_accuracy)
print("Macro F1 :", balanced_macro_f1)

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred_balanced
    )
)

In [ ]:
# ============================================================
# CONFUSION MATRIX - BALANCED MODEL
# ============================================================

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm_balanced = confusion_matrix(
    y_test,
    y_pred_balanced,
    labels=[
        "Healthy",
        "Moderate",
        "Needs Improvement"
    ]
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm_balanced,
    annot=True,
    fmt="d",
    cmap="viridis",
    xticklabels=[
        "Healthy",
        "Moderate",
        "Needs Improvement"
    ],
    yticklabels=[
        "Healthy",
        "Moderate",
        "Needs Improvement"
    ]
)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix - Balanced Gradient Boosting")

plt.show()

## comparision

In [ ]:
# ============================================================
# ORIGINAL vs BALANCED MODEL COMPARISON
# ============================================================

from sklearn.metrics import accuracy_score, f1_score

# Original model predictions
y_pred_original = best_gb.predict(X_test_encoded)

# Original model metrics
original_accuracy = accuracy_score(
    y_test,
    y_pred_original
)

original_macro_f1 = f1_score(
    y_test,
    y_pred_original,
    average="macro"
)

# Balanced model metrics
balanced_accuracy = accuracy_score(
    y_test,
    y_pred_balanced
)

balanced_macro_f1 = f1_score(
    y_test,
    y_pred_balanced,
    average="macro"
)

# Create comparison table
model_comparison = pd.DataFrame({
    "Model": [
        "Original Gradient Boosting",
        "Balanced Gradient Boosting"
    ],
    "Accuracy": [
        original_accuracy,
        balanced_accuracy
    ],
    "Macro F1": [
        original_macro_f1,
        balanced_macro_f1
    ]
})

print(model_comparison)

In [ ]:
# ============================================================
# FORMATTED MODEL COMPARISON
# ============================================================

comparison_display = model_comparison.copy()

comparison_display["Accuracy"] = (
    comparison_display["Accuracy"] * 100
).round(2)

comparison_display["Macro F1"] = (
    comparison_display["Macro F1"] * 100
).round(2)

comparison_display

In [ ]:
# ============================================================
# CLASS-WISE COMPARISON
# ORIGINAL vs BALANCED MODEL
# ============================================================

from sklearn.metrics import classification_report

# Original model report
original_report = classification_report(
    y_test,
    y_pred_original,
    output_dict=True
)

# Balanced model report
balanced_report = classification_report(
    y_test,
    y_pred_balanced,
    output_dict=True
)

classes = [
    "Healthy",
    "Moderate",
    "Needs Improvement"
]

class_comparison = pd.DataFrame({
    "Class": classes,

    "Original Precision": [
        original_report["Healthy"]["precision"],
        original_report["Moderate"]["precision"],
        original_report["Needs Improvement"]["precision"]
    ],

    "Balanced Precision": [
        balanced_report["Healthy"]["precision"],
        balanced_report["Moderate"]["precision"],
        balanced_report["Needs Improvement"]["precision"]
    ],

    "Original Recall": [
        original_report["Healthy"]["recall"],
        original_report["Moderate"]["recall"],
        original_report["Needs Improvement"]["recall"]
    ],

    "Balanced Recall": [
        balanced_report["Healthy"]["recall"],
        balanced_report["Moderate"]["recall"],
        balanced_report["Needs Improvement"]["recall"]
    ],

    "Original F1": [
        original_report["Healthy"]["f1-score"],
        original_report["Moderate"]["f1-score"],
        original_report["Needs Improvement"]["f1-score"]
    ],

    "Balanced F1": [
        balanced_report["Healthy"]["f1-score"],
        balanced_report["Moderate"]["f1-score"],
        balanced_report["Needs Improvement"]["f1-score"]
    ]
})

class_comparison.round(3)

### traing matrix

In [ ]:
# ============================================================
# TRAINING DATA PREDICTIONS
# ============================================================

# Predict on the training data using the already-trained model
y_pred_train = best_gb.predict(X_train_encoded)

print("Number of training predictions:", len(y_pred_train))

In [ ]:
# ============================================================
# TRAINING DATA EVALUATION
# ============================================================

from sklearn.metrics import accuracy_score, f1_score

train_accuracy = accuracy_score(
    y_train,
    y_pred_train
)

train_macro_f1 = f1_score(
    y_train,
    y_pred_train,
    average="macro"
)

print("Training Accuracy :", train_accuracy)
print("Training Macro F1  :", train_macro_f1)

In [ ]:
# ============================================================
# CONFUSION MATRIX - TRAINING DATA
# ============================================================

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm_train = confusion_matrix(
    y_train,
    y_pred_train,
    labels=[
        "Healthy",
        "Moderate",
        "Needs Improvement"
    ]
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm_train,
    annot=True,
    fmt="d",
    cmap="viridis",
    xticklabels=[
        "Healthy",
        "Moderate",
        "Needs Improvement"
    ],
    yticklabels=[
        "Healthy",
        "Moderate",
        "Needs Improvement"
    ]
)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix - Training Data")

plt.show()

# check expense

In [ ]:
# Check expense statuses for the test user

for category in expense_columns:
    status_column = category + "_Status"

    print(
        f"{category}: "
        f"{test_user[status_column].iloc[0]}"
    )

In [ ]:
# Display all expense recommendations

for recommendation in test_user["Expense_Recommendations"].iloc[0]:
    print("-", recommendation)